####Segment 1 - DELETE and UPDATE as standalone commands - internals

In [0]:
USE CATALOG workspace;
USE SCHEMA default;

####1. Create customers

In [0]:
DROP TABLE IF EXISTS customers;

CREATE OR REPLACE TABLE customers (
  customer_id  INT,
  name         STRING,
  email        STRING,
  tier         STRING,
  updated_at   TIMESTAMP
)
USING DELTA;

INSERT INTO customers VALUES
  (1, 'Alice Nguyen',  'alice@example.com',  'gold',     '2024-06-20 10:00:00'),
  (2, 'Bob Patel',     'bob@example.com',    'silver',   '2024-06-20 10:00:00'),
  (3, 'Carol Santos',  'carol@example.com',  'platinum', '2024-06-20 10:00:00'),
  (4, 'David Kim',     'david@example.com',  'bronze',   '2024-06-20 10:00:00'),
  (5, 'Eva Müller',    'eva@example.com',    'silver',   '2024-06-20 10:00:00'),
  (6, 'Frank Osei',    'frank@example.com',  'bronze',   '2024-06-20 10:00:00'),
  (7, 'Grace Lin',     'grace@example.com',  'gold',     '2024-06-20 10:00:00');

####2. Create erasure_requests for the GDPR

In [0]:
CREATE OR REPLACE TABLE erasure_requests (
  customer_id  INT,
  requested_at DATE
)
USING DELTA;

INSERT INTO erasure_requests VALUES
  (2, '2024-06-25'),
  (4, '2024-06-26'),
  (5, '2024-06-27');

####3. GDPR DELETE using a subquery

In [0]:
DELETE FROM customers
WHERE customer_id IN (
  SELECT customer_id FROM erasure_requests
);

> Delta does not modify existing Parquet files in place.

1. When you DELETE a row, it creates a deletion vector and completes the transaction (Applicable in delete, update, and merge).
2. Earlier it used copy-on-write.
3. The optimize is now baked into update, delete and merge statements as a separate but immediately performed transaction.

>The deleted customers are gone from any query against the current table. But here is the GDPR caveat you must know: 

1. The data is not yet truly erased. It is still accessible via time travel to older version.
2. For a complete GDPR erasure, you need two steps: first the DELETE, then VACUUM after the retention window

####Segment 2 - UPDATE

####1. Create customer_spend

In [0]:
CREATE OR REPLACE TABLE customer_spend (
  customer_id  INT,
  total_spend  DOUBLE
)
USING DELTA;

INSERT INTO customer_spend VALUES
  (1, 8200.00),   -- Alice: high spender
  (3, 12500.00),  -- Carol: very high spender
  (6, 350.00);    -- Frank: low spender

####2. UPDATE with a subquery

In [0]:
UPDATE customers
SET    tier = 'platinum'
WHERE  customer_id IN (
  SELECT customer_id
  FROM   customer_spend
  WHERE  total_spend > 5000
);

####Segment 3 — The duplicate problem - Idempotency

> Any batch pipeline writing to storage can execute more than once. Four common causes:

1. Job retries
2. Spark task speculation
3. Kinesis/Kafka at-least-once approach
4. Manual reruns

>With plain Parquet on storage, all four cases produce duplicates — silently, with no error.\
With Delta table and no idempotency options, the same thing happens 

>Delta's solution: tag each write with two options 
1. txnAppId - a unique name for your pipeline
2. txnVersion - a unique version number for this specific run.

>Before committing, Delta checks the transaction log for a previous commit with the same ID and version. If it finds one, it skips the write entirely. Exactly-once semantics, built into the Delta writer, no external coordination required.

####1. Create a fresh orders table

In [0]:
CREATE OR REPLACE TABLE orders (
  order_id    INT,
  customer_id INT,
  amount      DOUBLE,
  order_date  DATE
)
USING DELTA;

####2. Write batch with txnAppId and txnVersion

In [0]:
%python
batch_1 = [
    (1001, 1, 199.99, "2024-07-01"),
    (1002, 3, 449.00, "2024-07-01"),
    (1003, 6, 89.50,  "2024-07-01"),
]

df = spark.createDataFrame(batch_1, ["order_id", "customer_id", "amount", "order_date"]).selectExpr(
    "CAST(order_id AS INT) AS order_id",
    "CAST(customer_id AS INT) AS customer_id",
    "amount",
    "CAST(order_date AS DATE) AS order_date"
)

df.write \
  .format("delta") \
  .mode("append") \
  .option("txnAppId", "orders_daily_pipeline") \
  .option("txnVersion", 1) \
  .saveAsTable("orders")

count = spark.table("orders").count()
print(f"Row count after write: {count}")

####3. DESCRIBE HISTORY after repeated runs

In [0]:
%sql
select * from orders

In [0]:
DESCRIBE HISTORY orders;

####Summary
DELETE. Atomic, copy-on-write —> deletion vector -> Auto optimize : Delta writes new files and marks old ones removed.\
UPDATE. Same. 

Idempotent writes. Job retries and at-least-once delivery are facts of life. \
.option("txnAppId", "pipeline_name").option("txnVersion", run_id) gives exactly-once semantics for batch writes with no external coordination.